# Causal Evaluation of Probes

This is a simple tutorial showing you to collect activations from intervention-points in a model. We'll compare 1D DAS IIA on each layer and position for `block_output` in pythia-70M with logistic regression probing accuracy. The task we'll look at is gender prediction, where gendered names are used in templates like "[name] walked because", which elicits the associated gendered pronoun "he" or "she" as the next-token prediction for this model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frankaging/pyvene/blob/main/tutorials/advance_tutorials/Probing_Gender.ipynb)


In [ ]:
__author__ = "Aryaman Arora"
__version__ = "01/10/2024"

## Setup

In [4]:
!pip install nnsight

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.0/99.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 7.2 MB/s eta 0:00:00


In [5]:
try:
    # This library is our indicator that the required installs
    # need to be done.
    import pyvene as pv

except ModuleNotFoundError:
    !pip install git+https://github.com/stanfordnlp/pyvene.git

In [6]:
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
import torch
import random
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

%config InlineBackend.figure_formats = ['svg']
from plotnine import (
    ggplot,
    geom_tile,
    aes,
    facet_wrap,
    theme,
    element_text,
    geom_bar,
    geom_hline,
    scale_y_log10,
    geom_line,
    geom_point,
    geom_text,
    ggtitle, xlab, ylab,
    ggsave
)
from plotnine.scales import scale_y_reverse, scale_fill_cmap
from tqdm import tqdm
from collections import namedtuple

## Load model and data

In [7]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = "EleutherAI/pythia-70m" # "EleutherAI/pythia-6.9B"
tokenizer = AutoTokenizer.from_pretrained(model)
tokenizer.pad_token = tokenizer.eos_token
gpt = AutoModelForCausalLM.from_pretrained(
    model,
    revision="main",
    torch_dtype=torch.bfloat16 if model == "EleutherAI/pythia-6.9b" else torch.float32,
).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/166M [00:00<?, ?B/s]

In [17]:
from google.colab import output
output.enable_custom_widget_manager()

We have a list of 100 names for each gender, and we'll filter for names that are one token in length. We'll further filter for examples the model agrees with our labels for, since some of these names might be ambiguous or the model might not have the expected behaviour. This ensures that baseline IIA is 0.

In [8]:
Example = namedtuple("Example", ["base", "src", "base_label", "src_label"])

names = {
    "he": [
        "James",
        "Robert",
        "John",
        "Michael",
        "David",
        "William",
        "Richard",
        "Joseph",
        "Thomas",
        "Christopher",
        "Charles",
        "Daniel",
        "Matthew",
        "Anthony",
        "Mark",
        "Donald",
        "Steven",
        "Andrew",
        "Paul",
        "Joshua",
        "Kenneth",
        "Kevin",
        "Brian",
        "George",
        "Timothy",
        "Ronald",
        "Jason",
        "Edward",
        "Jeffrey",
        "Ryan",
        "Jacob",
        "Gary",
        "Nicholas",
        "Eric",
        "Jonathan",
        "Stephen",
        "Larry",
        "Justin",
        "Scott",
        "Brandon",
        "Benjamin",
        "Samuel",
        "Gregory",
        "Alexander",
        "Patrick",
        "Frank",
        "Raymond",
        "Jack",
        "Dennis",
        "Jerry",
        "Tyler",
        "Aaron",
        "Jose",
        "Adam",
        "Nathan",
        "Henry",
        "Zachary",
        "Douglas",
        "Peter",
        "Kyle",
        "Noah",
        "Ethan",
        "Jeremy",
        "Walter",
        "Christian",
        "Keith",
        "Roger",
        "Terry",
        "Austin",
        "Sean",
        "Gerald",
        "Carl",
        "Harold",
        "Dylan",
        "Arthur",
        "Lawrence",
        "Jordan",
        "Jesse",
        "Bryan",
        "Billy",
        "Bruce",
        "Gabriel",
        "Joe",
        "Logan",
        "Alan",
        "Juan",
        "Albert",
        "Willie",
        "Elijah",
        "Wayne",
        "Randy",
        "Vincent",
        "Mason",
        "Roy",
        "Ralph",
        "Bobby",
        "Russell",
        "Bradley",
        "Philip",
        "Eugene",
    ],
    "she": [
        "Mary",
        "Patricia",
        "Jennifer",
        "Linda",
        "Elizabeth",
        "Barbara",
        "Susan",
        "Jessica",
        "Sarah",
        "Karen",
        "Lisa",
        "Nancy",
        "Betty",
        "Sandra",
        "Margaret",
        "Ashley",
        "Kimberly",
        "Emily",
        "Donna",
        "Michelle",
        "Carol",
        "Amanda",
        "Melissa",
        "Deborah",
        "Stephanie",
        "Dorothy",
        "Rebecca",
        "Sharon",
        "Laura",
        "Cynthia",
        "Amy",
        "Kathleen",
        "Angela",
        "Shirley",
        "Brenda",
        "Emma",
        "Anna",
        "Pamela",
        "Nicole",
        "Samantha",
        "Katherine",
        "Christine",
        "Helen",
        "Debra",
        "Rachel",
        "Carolyn",
        "Janet",
        "Maria",
        "Catherine",
        "Heather",
        "Diane",
        "Olivia",
        "Julie",
        "Joyce",
        "Victoria",
        "Ruth",
        "Virginia",
        "Lauren",
        "Kelly",
        "Christina",
        "Joan",
        "Evelyn",
        "Judith",
        "Andrea",
        "Hannah",
        "Megan",
        "Cheryl",
        "Jacqueline",
        "Martha",
        "Madison",
        "Teresa",
        "Gloria",
        "Sara",
        "Janice",
        "Ann",
        "Kathryn",
        "Abigail",
        "Sophia",
        "Frances",
        "Jean",
        "Alice",
        "Judy",
        "Isabella",
        "Julia",
        "Grace",
        "Amber",
        "Denise",
        "Danielle",
        "Marilyn",
        "Beverly",
        "Charlotte",
        "Natalie",
        "Theresa",
        "Diana",
        "Brittany",
        "Doris",
        "Kayla",
        "Alexis",
        "Lori",
        "Marie",
    ],
}

# filter names that are > 1 token
names = {
    key: [name for name in names[key] if len(tokenizer.tokenize(name)) == 1]
    for key in names
}
print(len(names["he"]), len(names["she"]))


def sample_example(tokenizer):
    # sample labels (not matching)
    base_label = random.choice(list(names.keys()))
    src_label = [key for key in names if key != base_label][0]

    # sample names
    base_name = random.choice(names[base_label])
    src_name = random.choice(names[src_label])

    # make pair
    base = tokenizer(f"<|endoftext|>{base_name} walked because", return_tensors="pt")
    src = tokenizer(f"<|endoftext|>{src_name} walked because", return_tensors="pt")
    base_label = tokenizer.encode(" " + base_label)[0]
    src_label = tokenizer.encode(" " + src_label)[0]
    return Example(base, src, base_label, src_label)

47 10


In [9]:
sample_example(tokenizer)

Example(base={'input_ids': tensor([[    0, 47413,  7428,   984]]), 'attention_mask': tensor([[1, 1, 1, 1]])}, src={'input_ids': tensor([[    0, 49062,  7428,   984]]), 'attention_mask': tensor([[1, 1, 1, 1]])}, base_label=344, src_label=703)

In [10]:
def generate_n_doable_examples(n, model, tokenizer):
    examples = []
    iterator = tqdm(range(n))
    while len(examples) < n:
        ex = sample_example(tokenizer)
        for k, v in ex.base.items():
            if v is not None and isinstance(v, torch.Tensor):
                ex.base[k] = v.to(model.device)
        for k, v in ex.src.items():
            if v is not None and isinstance(v, torch.Tensor):
                ex.src[k] = v.to(model.device)
        logits_base = model(**ex.base).logits[0, -1]
        logits_src = model(**ex.src).logits[0, -1]
        if (
            logits_base[ex.base_label] > logits_base[ex.src_label]
            and logits_src[ex.src_label] > logits_src[ex.base_label]
        ):
            examples.append(ex)
            iterator.update(1)
    return examples

In [11]:
# make dataset
total_steps = 100
trainset = generate_n_doable_examples(total_steps, gpt, tokenizer)
evalset = generate_n_doable_examples(50, gpt, tokenizer)

100%|██████████| 50/50 [00:00<00:00, 68.48it/s]


## DAS

This is the usual 1D DAS setup, training on batch size of 1.

In [12]:
def intervention_config(intervention_site, layer, num_dims=1):
    config = pv.IntervenableConfig([
        {
            "layer": layer,
            "component": intervention_site,
            "intervention_type": pv.LowRankRotatedSpaceIntervention,
            "low_rank_dimension": num_dims,
        }
    ])
    return config

In [13]:
# loss function
loss_fct = torch.nn.CrossEntropyLoss()

def calculate_loss(logits, label):
    """Calculate cross entropy between logits and a single target label (can be batched)"""
    shift_labels = label.to(logits.device)
    loss = loss_fct(logits, shift_labels)
    return loss

In [14]:
# intervention settings
stats = []
num_layers = gpt.config.num_hidden_layers

# loop over layers and positions
for layer in range(num_layers):
    for position in range(4):
        print(f"layer: {layer}, position: {position}")

        # set up intervenable model
        config = intervention_config("block_output", layer, 1)
        intervenable = pv.IntervenableModel(config, gpt)
        intervenable.set_device(device)
        intervenable.disable_model_gradients()

        # set up optimizer
        optimizer_params = []
        for k, v in intervenable.interventions.items():
            try:
                optimizer_params.append({"params": v.rotate_layer.parameters()})
            except:
                pass
        optimizer = torch.optim.Adam(optimizer_params, lr=1e-3)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps,
        )

        # training loop
        iterator = tqdm(trainset)
        for example in iterator:
            # forward pass
            _, counterfactual_outputs = intervenable(
                example.base,
                [example.src],
                {"sources->base": position},
            )

            # loss
            logits = counterfactual_outputs.logits[:, -1]
            loss = calculate_loss(logits, torch.tensor([example.src_label]).to(device))
            iterator.set_postfix({"loss": f"{loss.item():.3f}"})

            # backward
            loss.backward()
            optimizer.step()
            scheduler.step()

        # eval
        with torch.no_grad():
            iia = 0
            iterator = tqdm(evalset)
            for example in iterator:
                # forward
                _, counterfactual_outputs = intervenable(
                    example.base,
                    [example.src],
                    {"sources->base": position},
                )

                # calculate iia
                logits = counterfactual_outputs.logits[0, -1]
                if logits[example.src_label] > logits[example.base_label]:
                    iia += 1

            # stats
            iia = iia / len(evalset)
            stats.append({"layer": layer, "position": position, "iia": iia})
            print(f"iia: {iia:.3%}")
df = pd.DataFrame(stats)
df.to_csv(f"./tutorial_data/pyvene_gender_das.csv")

layer: 0, position: 0


100%|██████████| 50/50 [00:00<00:00, 71.08it/s]


iia: 0.000%
layer: 0, position: 1


100%|██████████| 50/50 [00:00<00:00, 70.77it/s]


iia: 90.000%
layer: 0, position: 2


...

100%|██████████| 50/50 [00:00<00:00, 71.09it/s]


iia: 0.000%
layer: 5, position: 3


100%|██████████| 50/50 [00:00<00:00, 71.16it/s]

iia: 90.000%


OSError: Cannot save file into a non-existent directory: 'tutorial_data'

In [15]:
df.to_csv(f"./tutorial_data/pyvene_gender_das.csv")

And this is the plot of IIA. In layers 2 and 3 it seems the gender is represented across positions 1-3, and entirely in position 3 in later layers.

In [18]:
df = pd.read_csv(f"./tutorial_data/pyvene_gender_das.csv")
df["layer"] = df["layer"].astype(int)
df["pos"] = df["position"].astype(int)
df["IIA"] = df["iia"].astype(float)

custom_labels = ["EOS", "<name>", "walked", "because"]
breaks = [0, 1, 2, 3]

plot = (
    ggplot(df, aes(x="layer", y="pos"))
    + geom_tile(aes(fill="IIA"))
    + scale_fill_cmap("Purples") + xlab("layers")
    + scale_y_reverse(
        limits = (-0.5, 3.5),
        breaks=breaks, labels=custom_labels)
    + theme(figure_size=(5, 3)) + ylab("")
    + theme(axis_text_y  = element_text(angle = 90, hjust = 1))
    + ggtitle("Trained Intervention (DAS)")
)
ggsave(
    plot, filename=f"./tutorial_data/pyvene_gender_das.pdf", dpi=200
)
print(plot)

/usr/local/lib/python3.12/dist-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 3 in image.
/usr/local/lib/python3.12/dist-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: ./tutorial_data/pyvene_gender_das.pdf


<ggplot: (500 x 300)>


## Probing

We'll define a dummy intervention `CollectActivation` to collect activations and train a simple probe.

In [21]:
def probing_config(intervention_site, layer):
    """Generate intervention config."""

    # init
    config = pv.IntervenableConfig([{
        "layer": layer,
        "component": intervention_site,
        "intervention_type": pv.CollectIntervention,
    }])
    return config

This is the training loop.

In [24]:
# intervention settings
stats = []
num_layers = gpt.config.num_hidden_layers

# 344 = " he", 703 = " she"
label_mapping = {344: 0, 703: 1}

# loop over layers and positions
with torch.no_grad():
    for layer in range(num_layers):
        for position in range(4):
            print(f"layer: {layer}, position: {position}")

            # set up intervenable model
            config = probing_config("block_output", layer)
            intervenable = pv.IntervenableModel(config, gpt)
            intervenable.set_device(device)
            intervenable.disable_model_gradients()

            # training loop
            activations, labels = [], []
            iterator = tqdm(trainset)
            for example in iterator:
                # forward pass
                base_outputs, _ = intervenable(
                    example.base,
                    unit_locations={"base": position},
                )
                base_activations = base_outputs[1][0]

                src_outputs, _ = intervenable(
                    example.src,
                    unit_locations={"base": position},
                )
                src_activations = src_outputs[1][0]

                # collect activation
                activations.extend(
                    [base_activations.squeeze().detach().cpu().numpy(), src_activations.squeeze().detach().cpu().numpy()]
                )
                labels.extend([example.base_label, example.src_label])
            labels = [label_mapping[label] for label in labels]

            # train logistic regression
            lr = LogisticRegression(random_state=42, max_iter=1000).fit(
                activations, labels
            )

            # eval
            activations, labels = [], []
            iterator = tqdm(evalset)
            for example in iterator:
                # forward pass
                base_outputs, _ = intervenable(
                    example.base,
                    unit_locations={"base": position},
                )
                base_activations = base_outputs[1][0]

                src_outputs, _ = intervenable(
                    example.src,
                    unit_locations={"base": position},
                )
                src_activations = src_outputs[1][0]

                # collect activation
                activations.extend(
                    [base_activations.squeeze().detach().cpu().numpy(), src_activations.squeeze().detach().cpu().numpy()]
                )
                labels.extend([example.base_label, example.src_label])
            labels = [label_mapping[label] for label in labels]

            # stats
            acc = lr.score(activations, labels)
            f1 = f1_score(labels, lr.predict(activations))
            stats.append({"layer": layer, "position": position, "acc": acc, "f1": f1})
            print(f"acc: {acc:.3%}, f1: {f1:.3f}")
df = pd.DataFrame(stats)
df.to_csv(f"./tutorial_data/pyvene_gender_probe.csv")

layer: 0, position: 0


100%|██████████| 50/50 [00:01<00:00, 29.04it/s]


acc: 50.000%, f1: 0.000
layer: 0, position: 1


100%|██████████| 50/50 [00:01<00:00, 40.55it/s]


acc: 100.000%, f1: 1.000
layer: 0, position: 2


...

acc: 100.000%, f1: 1.000
layer: 5, position: 2


100%|██████████| 50/50 [00:00<00:00, 77.67it/s]


acc: 100.000%, f1: 1.000
layer: 5, position: 3


100%|██████████| 50/50 [00:00<00:00, 78.42it/s]

acc: 100.000%, f1: 1.000


And the probe accuracy plot is below. Note the extremely high accuracy at all positions at and after the name! Early layers at later positions are better but it saturates much before the IIA for DAS. This shows how unreliable probes are for tracing causal effect.

In [25]:
df = pd.read_csv(f"./tutorial_data/pyvene_gender_probe.csv")
df["layer"] = df["layer"].astype(int)
df["pos"] = df["position"].astype(int)
df["ACC"] = df["acc"].astype(float)

custom_labels = ["EOS", "<name>", "walked", "because"]
breaks = [0, 1, 2, 3]

plot = (
    ggplot(df, aes(x="layer", y="pos", fill="ACC"))
    + geom_tile()
    + scale_fill_cmap("Reds") + xlab("layers")
    + scale_y_reverse(
        limits = (-0.5, 3.5),
        breaks=breaks, labels=custom_labels)
    + theme(figure_size=(5, 3)) + ylab("")
    + theme(axis_text_y  = element_text(angle = 90, hjust = 1))
    + ggtitle("Trained Linear Probe")
)
ggsave(
    plot, filename=f"./tutorial_data/pyvene_gender_probe.pdf", dpi=200
)
print(plot)

<ggplot: (500 x 300)>


/usr/local/lib/python3.12/dist-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 3 in image.
/usr/local/lib/python3.12/dist-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: ./tutorial_data/pyvene_gender_probe.pdf


# Agents-motivations

In [26]:
!pip install datasets

In [39]:
from datasets import load_dataset

babi_nli_dataset = load_dataset("tasksource/babi_nli", "agents-motivations")
print(babi_nli_dataset)

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 500
    })
    test: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 500
    })
})


In [40]:
print(babi_nli_dataset["train"][0])
print(babi_nli_dataset["train"][1])
print(babi_nli_dataset["train"][2])


{'premise': 'Sumit is bored. Jason is bored. Yann is thirsty. Jason went to the garden. Jason grabbed the football there. Yann moved to the kitchen. Yann got the milk there. Sumit moved to the garden.', 'hypothesis': 'Sumit went to the garden because she was bored.', 'label': 1, 'idx': 0}
{'premise': 'Antoine is hungry. Antoine went back to the kitchen. Antoine picked up the apple there. Jason is tired. Yann is thirsty. Sumit is bored. Yann went back to the kitchen. Yann picked up the milk there. Jason went to the bedroom. Jason picked up the pajamas there. Sumit went to the garden.', 'hypothesis': 'Sumit went to the garden because she was tired.', 'label': 0, 'idx': 1}
{'premise': 'Antoine is thirsty. Sumit is thirsty. Jason is thirsty. Sumit journeyed to the kitchen. Jason travelled to the kitchen. Yann is bored.', 'hypothesis': 'Yann will go to the kitchen.', 'label': 0, 'idx': 2}


In [ ]:
import re

def extract_room_info(example):
    hypothesis = example["hypothesis"]
    match = re.search(r"to the (\w+)", hypothesis)
    if match:
        room_name = match.group(1)
        return {"extracted_text": hypothesis, "extracted_room": room_name.lower()}
    else:
        return {"extracted_text": hypothesis, "extracted_room": None}


processed_train_data = babi_nli_dataset["train"].map(extract_room_info)
processed_validation_data = babi_nli_dataset["validation"].map(extract_room_info)
processed_test_data = babi_nli_dataset["test"].map(extract_room_info)

processed_train_data = processed_train_data.filter(lambda x: x["extracted_room"] is not None)
processed_validation_data = processed_validation_data.filter(lambda x: x["extracted_room"] is not None)
processed_test_data = processed_test_data.filter(lambda x: x["extracted_room"] is not None)

columns_to_keep = ["extracted_text", "extracted_room"]
all_current_columns_train = processed_train_data.column_names
cols_to_drop_train = [col for col in all_current_columns_train if col not in columns_to_keep]

all_current_columns_val = processed_validation_data.column_names
cols_to_drop_val = [col for col in all_current_columns_val if col not in columns_to_keep]

all_current_columns_test = processed_test_data.column_names
cols_to_drop_test = [col for col in all_current_columns_test if col not in columns_to_keep]

processed_train_data = processed_train_data.remove_columns(cols_to_drop_train)
processed_validation_data = processed_validation_data.remove_columns(cols_to_drop_val)
processed_test_data = processed_test_data.remove_columns(cols_to_drop_test)

processed_train_data = processed_train_data.rename_columns({"extracted_text": "text", "extracted_room": "room"})
processed_validation_data = processed_validation_data.rename_columns({"extracted_text": "text", "extracted_room": "room"})
processed_test_data = processed_test_data.rename_columns({"extracted_text": "text", "extracted_room": "room"})


print("Processed Train Data Sample:", processed_train_data[0])
print("Processed Validation Data Sample:", processed_validation_data[0])
print("Processed Test Data Sample:", processed_test_data[0])

Processed Train Data Sample: {'text': 'Sumit went to the garden because she was bored.', 'room': 'garden'}
Processed Validation Data Sample: {'text': 'Antoine will go to the kitchen.', 'room': 'kitchen'}
Processed Test Data Sample: {'text': 'Jason went to the kitchen because she was tired.', 'room': 'kitchen'}


In [42]:
unique_rooms = set(processed_train_data["room"]) | set(processed_validation_data["room"]) | set(processed_test_data["room"])
room_to_label = {room: i for i, room in enumerate(sorted(list(unique_rooms)))}
label_to_room = {i: room for room, i in room_to_label.items()}

def map_room_to_label(example):
    example["room_label"] = room_to_label[example["room"]]
    return example

processed_train_data = processed_train_data.map(map_room_to_label)
processed_validation_data = processed_validation_data.map(map_room_to_label)
processed_test_data = processed_test_data.map(map_room_to_label)

print("Room to Label Mapping:", room_to_label)
print("Processed Train Data Sample with labels:", processed_train_data[0])
print("Processed Validation Data Sample with labels:", processed_validation_data[0])
print("Processed Test Data Sample with labels:", processed_test_data[0])

Room to Label Mapping: {'bedroom': 0, 'garden': 1, 'kitchen': 2}
Processed Train Data Sample with labels: {'text': 'Sumit went to the garden because she was bored.', 'room': 'garden', 'room_label': 1}
Processed Validation Data Sample with labels: {'text': 'Antoine will go to the kitchen.', 'room': 'kitchen', 'room_label': 2}
Processed Test Data Sample with labels: {'text': 'Jason went to the kitchen because she was tired.', 'room': 'kitchen', 'room_label': 2}


In [ ]:
RoomExample = namedtuple("RoomExample", ["input_ids", "attention_mask", "label"])

def generate_room_classification_examples(dataset, tokenizer, num_examples):
    examples = []
    num_iter = 0
    for entry in dataset:
        text = entry["text"]
        room_label = entry["room_label"]

        tokenized_input = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=gpt.config.max_position_embeddings)

        examples.append(
            RoomExample(
                input_ids=tokenized_input["input_ids"].squeeze(),
                attention_mask=tokenized_input["attention_mask"].squeeze(),
                label=torch.tensor(room_label)
            )
        )
        num_iter += 1
        if num_iter >= num_examples:
            break
    return examples

train_examples = generate_room_classification_examples(processed_train_data, tokenizer, 100)
validation_examples = generate_room_classification_examples(processed_validation_data, tokenizer, 50)

print(f"Generated {len(train_examples)} training examples.")
print(f"Generated {len(validation_examples)} validation examples.")
print("Sample RoomExample:", train_examples[0])

Generated 100 training examples.
Generated 50 validation examples.
Sample RoomExample: RoomExample(input_ids=tensor([11808,   262,  2427,  ...,     0,     0,     0]), attention_mask=tensor([1, 1, 1,  ..., 0, 0, 0]), label=tensor(1))


In [ ]:
stats_room = []
num_layers = gpt.config.num_hidden_layers

with torch.no_grad():
    for layer in range(num_layers):
        for position in range(gpt.config.max_position_embeddings):
            if position >= 4:
                break

            print(f"layer: {layer}, position: {position}")

            config = probing_config("block_output", layer)
            intervenable = pv.IntervenableModel(config, gpt)
            intervenable.set_device(device)
            intervenable.disable_model_gradients()

            activations_train, labels_train = [], []
            iterator_train = tqdm(train_examples, desc=f"Collecting train data for L{layer} P{position}")
            for example in iterator_train:
                inputs = {
                    "input_ids": example.input_ids.unsqueeze(0).to(device),
                    "attention_mask": example.attention_mask.unsqueeze(0).to(device)
                }
                outputs, _ = intervenable(
                    inputs,
                    unit_locations={"base": position},
                )
                activations_train.append(outputs[1][0].squeeze().detach().cpu().numpy())
                labels_train.append(example.label.item())

            labels_train = [int(lbl) for lbl in labels_train]


            if len(set(labels_train)) > 1 and len(activations_train) > 0:
                lr = LogisticRegression(random_state=42, max_iter=1000).fit(
                    activations_train, labels_train
                )
            else:
                print(f"Skipping training for L{layer} P{position} due to insufficient classes or data.")
                lr = None 

            activations_eval, labels_eval = [], []
            iterator_eval = tqdm(validation_examples, desc=f"Collecting eval data for L{layer} P{position}")
            for example in iterator_eval:
                inputs = {
                    "input_ids": example.input_ids.unsqueeze(0).to(device),
                    "attention_mask": example.attention_mask.unsqueeze(0).to(device)
                }
                outputs, _ = intervenable(
                    inputs,
                    unit_locations={"base": position},
                )
                activations_eval.append(outputs[1][0].squeeze().detach().cpu().numpy())
                labels_eval.append(example.label.item())

            labels_eval = [int(lbl) for lbl in labels_eval]

            if lr:
                acc = lr.score(activations_eval, labels_eval)
                f1 = f1_score(labels_eval, lr.predict(activations_eval), average='weighted')
            else:
                acc = 0.0
                f1 = 0.0

            stats_room.append({"layer": layer, "position": position, "acc": acc, "f1": f1})
            print(f"acc: {acc:.3%}, f1: {f1:.3f}")

df_room_probe = pd.DataFrame(stats_room)
df_room_probe.to_csv(f"./tutorial_data/pyvene_room_probe.csv", index=False)
print("Room probing results saved to ./tutorial_data/pyvene_room_probe.csv")


layer: 0, position: 0


acc: 38.000%, f1: 0.209
layer: 0, position: 1


acc: 38.000%, f1: 0.242
layer: 0, position: 2


acc: 40.000%, f1: 0.308
layer: 0, position: 3


acc: 40.000%, f1: 0.308
layer: 1, position: 0


...

acc: 40.000%, f1: 0.308
layer: 5, position: 0


acc: 38.000%, f1: 0.209
layer: 5, position: 1


acc: 38.000%, f1: 0.242
layer: 5, position: 2


acc: 40.000%, f1: 0.308
layer: 5, position: 3


acc: 40.000%, f1: 0.308
Room probing results saved to ./tutorial_data/pyvene_room_probe.csv


In [ ]:
RoomExample = namedtuple("RoomExample", ["input_ids", "attention_mask", "label"])

def generate_room_classification_examples(dataset, tokenizer, num_examples):
    examples = []
    num_iter = 0
    for entry in dataset:
        text = entry["text"]
        room_label = entry["room_label"]

        tokenized_input = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=gpt.config.max_position_embeddings)

        examples.append(
            RoomExample(
                input_ids=tokenized_input["input_ids"].squeeze(),
                attention_mask=tokenized_input["attention_mask"].squeeze(),
                label=torch.tensor(room_label)
            )
        )
        num_iter += 1
        if num_iter >= num_examples:
            break
    return examples

train_examples = generate_room_classification_examples(processed_train_data, tokenizer, 100)
validation_examples = generate_room_classification_examples(processed_validation_data, tokenizer, 50)

print(f"Generated {len(train_examples)} training examples.")
print(f"Generated {len(validation_examples)} validation examples.")
print("Sample RoomExample:", train_examples[0])

Generated 100 training examples.
Generated 50 validation examples.
Sample RoomExample: RoomExample(input_ids=tensor([11808,   262,  2427,  ...,     0,     0,     0]), attention_mask=tensor([1, 1, 1,  ..., 0, 0, 0]), label=tensor(1))


In [ ]:
import os

if not os.path.exists('./tutorial_data'):
    os.makedirs('./tutorial_data')

df_room_probe["layer"] = df_room_probe["layer"].astype(int)
df_room_probe["pos"] = df_room_probe["position"].astype(int)
df_room_probe["ACC"] = df_room_probe["acc"].astype(float)


custom_labels_room = [str(i) for i in range(4)]
breaks_room = [0, 1, 2, 3]

plot_room = (
    ggplot(df_room_probe, aes(x="layer", y="pos", fill="ACC"))
    + geom_tile()
    + scale_fill_cmap("Reds") + xlab("layers")
    + scale_y_reverse(
        limits = (-0.5, 3.5),
        breaks=breaks_room, labels=custom_labels_room)
    + theme(figure_size=(5, 3)) + ylab("Position in Sentence")
    + theme(axis_text_y  = element_text(angle = 90, hjust = 1))
    + ggtitle("Trained Linear Probe (Room Classification)")
)
ggsave(
    plot_room, filename=f"./tutorial_data/pyvene_room_probe.pdf", dpi=200
)
print(plot_room)

<ggplot: (500 x 300)>


/usr/local/lib/python3.12/dist-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 3 in image.
/usr/local/lib/python3.12/dist-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: ./tutorial_data/pyvene_room_probe.pdf


# Compound-coreference

In [52]:
from datasets import load_dataset

babi_nli_coreference_dataset = load_dataset("tasksource/babi_nli", "compound-coreference")
print(babi_nli_coreference_dataset)

print("\nSample 1:", babi_nli_coreference_dataset["train"][0])
print("Sample 2:", babi_nli_coreference_dataset["train"][1])
print("Sample 3:", babi_nli_coreference_dataset["train"][2])

compound-coreference/train-00000-of-0000(…):   0%|          | 0.00/49.8k [00:00<?, ?B/s]

compound-coreference/validation-00000-of(…):   0%|          | 0.00/29.4k [00:00<?, ?B/s]

compound-coreference/test-00000-of-00001(…):   0%|          | 0.00/30.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 500
    })
    test: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 500
    })
})

Sample 1: {'premise': 'John and Daniel went to the office. After that they went back to the kitchen.', 'hypothesis': 'John is in the kitchen.', 'label': 1, 'idx': 0}
Sample 2: {'premise': 'Daniel and Sandra went to the office. Following that they travelled to the hallway.', 'hypothesis': 'Sandra is in the office.', 'label': 0, 'idx': 1}
Sample 3: {'premise': 'John and Sandra journeyed to the bedroom. Afterwards they moved to the kitchen. Daniel and Sandra moved to the bedroom. Then they moved to the bathroom. John and Daniel went back to the garden. Then they went back to the office.', 'hypothesis': 'Daniel is in the kitchen.', 'label':

In [ ]:
import re

def extract_room_info_coreference(example):
    if example['label'] != 1:
        return {"extracted_text": None, "extracted_room": None}

    hypothesis = example["hypothesis"]
    match_ = re.search(r"(.*?)\s+in the\s+(\w+)\.?", hypothesis)

    if match_:
        person_name = match_.group(1)
        room_name = match_.group(2).lower()
        constructed_text = f"{example['premise']} {person_name} is in the "
        return {"extracted_text": constructed_text, "extracted_room": room_name}
    else:
        return {"extracted_text": None, "extracted_room": None}

processed_train_data_coref = babi_nli_coreference_dataset["train"].map(extract_room_info_coreference)
processed_validation_data_coref = babi_nli_coreference_dataset["validation"].map(extract_room_info_coreference)
processed_test_data_coref = babi_nli_coreference_dataset["test"].map(extract_room_info_coreference)

processed_train_data_coref = processed_train_data_coref.filter(lambda x: x["extracted_room"] is not None)
processed_validation_data_coref = processed_validation_data_coref.filter(lambda x: x["extracted_room"] is not None)
processed_test_data_coref = processed_test_data_coref.filter(lambda x: x["extracted_room"] is not None)

print(f"Train examples after filtering: {len(processed_train_data_coref)}")
print(f"Validation examples after filtering: {len(processed_validation_data_coref)}")
print(f"Test examples after filtering: {len(processed_test_data_coref)}")

columns_to_keep_coref = ["extracted_text", "extracted_room"]

if len(processed_train_data_coref) > 0:
    all_current_columns_train_coref = processed_train_data_coref.column_names
    cols_to_drop_train_coref = [col for col in all_current_columns_train_coref if col not in columns_to_keep_coref]
    processed_train_data_coref = processed_train_data_coref.remove_columns(cols_to_drop_train_coref)
    processed_train_data_coref = processed_train_data_coref.rename_columns({"extracted_text": "text", "extracted_room": "room"})
    print("Processed Train Data (Compound Coreference) Sample:", processed_train_data_coref[0])
else:
    print("No processed train data to display sample.")

if len(processed_validation_data_coref) > 0:
    all_current_columns_val_coref = processed_validation_data_coref.column_names
    cols_to_drop_val_coref = [col for col in all_current_columns_val_coref if col not in columns_to_keep_coref]
    processed_validation_data_coref = processed_validation_data_coref.remove_columns(cols_to_drop_val_coref)
    processed_validation_data_coref = processed_validation_data_coref.rename_columns({"extracted_text": "text", "extracted_room": "room"})
    print("Processed Validation Data (Compound Coreference) Sample:", processed_validation_data_coref[0])
else:
    print("No processed validation data to display sample.")

if len(processed_test_data_coref) > 0:
    all_current_columns_test_coref = processed_test_data_coref.column_names
    cols_to_drop_test_coref = [col for col in all_current_columns_test_coref if col not in columns_to_keep_coref]
    processed_test_data_coref = processed_test_data_coref.remove_columns(cols_to_drop_test_coref)
    processed_test_data_coref = processed_test_data_coref.rename_columns({"extracted_text": "text", "extracted_room": "room"})
    print("Processed Test Data (Compound Coreference) Sample:", processed_test_data_coref[0])
else:
    print("No processed test data to display sample.")


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Train examples after filtering: 503
Validation examples after filtering: 272
Test examples after filtering: 231
Processed Train Data (Compound Coreference) Sample: {'text': 'John and Daniel went to the office. After that they went back to the kitchen. John is is in the ', 'room': 'kitchen'}
Processed Validation Data (Compound Coreference) Sample: {'text': 'Daniel and Sandra moved to the kitchen. Afterwards they travelled to the hallway. John and Sandra travelled to the bathroom. Then they journeyed to the garden. Sandra is is in the ', 'room': 'garden'}
Processed Test Data (Compound Coreference) Sample: {'text': 'Daniel and John moved to the office. After that they went back to the bathroom. Daniel and Mary went back to the garden. After that they journeyed to the bathroom. Mary and John journeyed to the hallway. Then they went to the bedroom. Daniel and John journeyed to the office. After that they went back to the hallway. Sandra and Daniel moved to the kitchen. Following that they m

In [55]:
unique_rooms_coref = set(processed_train_data_coref["room"]) | set(processed_validation_data_coref["room"]) | set(processed_test_data_coref["room"])
room_to_label_coref = {room: i for i, room in enumerate(sorted(list(unique_rooms_coref)))}
label_to_room_coref = {i: room for room, i in room_to_label_coref.items()}

def map_room_to_label_coref(example):
    example["room_label"] = room_to_label_coref[example["room"]]
    return example

processed_train_data_coref = processed_train_data_coref.map(map_room_to_label_coref)
processed_validation_data_coref = processed_validation_data_coref.map(map_room_to_label_coref)
processed_test_data_coref = processed_test_data_coref.map(map_room_to_label_coref)

print("Room to Label Mapping (Compound Coreference):", room_to_label_coref)
print("Processed Train Data Sample with labels (Compound Coreference):", processed_train_data_coref[0])
print("Processed Validation Data Sample with labels (Compound Coreference):", processed_validation_data_coref[0])
print("Processed Test Data Sample with labels (Compound Coreference):", processed_test_data_coref[0])

Map:   0%|          | 0/503 [00:00<?, ? examples/s]

Map:   0%|          | 0/272 [00:00<?, ? examples/s]

Map:   0%|          | 0/231 [00:00<?, ? examples/s]

Room to Label Mapping (Compound Coreference): {'bathroom': 0, 'bedroom': 1, 'garden': 2, 'hallway': 3, 'kitchen': 4, 'office': 5}
Processed Train Data Sample with labels (Compound Coreference): {'text': 'John and Daniel went to the office. After that they went back to the kitchen. John is is in the ', 'room': 'kitchen', 'room_label': 4}
Processed Validation Data Sample with labels (Compound Coreference): {'text': 'Daniel and Sandra moved to the kitchen. Afterwards they travelled to the hallway. John and Sandra travelled to the bathroom. Then they journeyed to the garden. Sandra is is in the ', 'room': 'garden', 'room_label': 2}
Processed Test Data Sample with labels (Compound Coreference): {'text': 'Daniel and John moved to the office. After that they went back to the bathroom. Daniel and Mary went back to the garden. After that they journeyed to the bathroom. Mary and John journeyed to the hallway. Then they went to the bedroom. Daniel and John journeyed to the office. After that they

In [ ]:
RoomExample = namedtuple("RoomExample", ["input_ids", "attention_mask", "label"])

def generate_room_classification_examples_coref(dataset, tokenizer, num_examples):
    examples = []
    num_iter = 0
    for entry in dataset:
        text = entry["text"]
        room_label = entry["room_label"]

        tokenized_input = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=gpt.config.max_position_embeddings)

        examples.append(
            RoomExample(
                input_ids=tokenized_input["input_ids"].squeeze(),
                attention_mask=tokenized_input["attention_mask"].squeeze(),
                label=torch.tensor(room_label)
            )
        )
        num_iter += 1
        if num_iter >= num_examples:
            break
    return examples

train_examples_coref = generate_room_classification_examples_coref(processed_train_data_coref, tokenizer, 100)
validation_examples_coref = generate_room_classification_examples_coref(processed_validation_data_coref, tokenizer, 50)

print(f"Generated {len(train_examples_coref)} training examples for compound coreference.")
print(f"Generated {len(validation_examples_coref)} validation examples for compound coreference.")
print("Sample RoomExample (Compound Coreference):", train_examples_coref[0])

Generated 100 training examples for compound coreference.
Generated 50 validation examples for compound coreference.
Sample RoomExample (Compound Coreference): RoomExample(input_ids=tensor([ 8732,   285, 10213,  ...,     0,     0,     0]), attention_mask=tensor([1, 1, 1,  ..., 0, 0, 0]), label=tensor(4))


In [ ]:
stats_room_coref = []
num_layers = gpt.config.num_hidden_layers

with torch.no_grad():
    for layer in range(num_layers):
        for position in range(gpt.config.max_position_embeddings):
            if position >= 4:
                break

            print(f"layer: {layer}, position: {position}")

            config = probing_config("block_output", layer)
            intervenable = pv.IntervenableModel(config, gpt)
            intervenable.set_device(device)
            intervenable.disable_model_gradients()

            activations_train_coref, labels_train_coref = [], []
            iterator_train_coref = tqdm(train_examples_coref, desc=f"Collecting train data for L{layer} P{position} (Coref)")
            for example in iterator_train_coref:
                inputs = {
                    "input_ids": example.input_ids.unsqueeze(0).to(device),
                    "attention_mask": example.attention_mask.unsqueeze(0).to(device)
                }
                outputs, _ = intervenable(
                    inputs,
                    unit_locations={"base": position},
                )
                activations_train_coref.append(outputs[1][0].squeeze().detach().cpu().numpy())
                labels_train_coref.append(example.label.item())

            labels_train_coref = [int(lbl) for lbl in labels_train_coref]

            if len(set(labels_train_coref)) > 1 and len(activations_train_coref) > 0:
                lr_coref = LogisticRegression(random_state=42, max_iter=1000).fit(
                    activations_train_coref, labels_train_coref
                )
            else:
                print(f"Skipping training for L{layer} P{position} (Coref) due to insufficient classes or data.")
                lr_coref = None

            activations_eval_coref, labels_eval_coref = [], []
            iterator_eval_coref = tqdm(validation_examples_coref, desc=f"Collecting eval data for L{layer} P{position} (Coref)")
            for example in iterator_eval_coref:
                inputs = {
                    "input_ids": example.input_ids.unsqueeze(0).to(device),
                    "attention_mask": example.attention_mask.unsqueeze(0).to(device)
                }
                outputs, _ = intervenable(
                    inputs,
                    unit_locations={"base": position},
                )
                activations_eval_coref.append(outputs[1][0].squeeze().detach().cpu().numpy())
                labels_eval_coref.append(example.label.item())

            labels_eval_coref = [int(lbl) for lbl in labels_eval_coref]

            if lr_coref:
                acc_coref = lr_coref.score(activations_eval_coref, labels_eval_coref)
                f1_coref = f1_score(labels_eval_coref, lr_coref.predict(activations_eval_coref), average='weighted')
            else:
                acc_coref = 0.0
                f1_coref = 0.0

            stats_room_coref.append({"layer": layer, "position": position, "acc": acc_coref, "f1": f1_coref})
            print(f"acc (Coref): {acc_coref:.3%}, f1 (Coref): {f1_coref:.3f}")

df_room_probe_coref = pd.DataFrame(stats_room_coref)
df_room_probe_coref.to_csv(f"./tutorial_data/pyvene_room_probe_coref.csv", index=False)
print("Compound coreference room probing results saved to ./tutorial_data/pyvene_room_probe_coref.csv")

layer: 0, position: 0


acc (Coref): 14.000%, f1 (Coref): 0.084
layer: 0, position: 1


acc (Coref): 12.000%, f1 (Coref): 0.089
layer: 0, position: 2


acc (Coref): 12.000%, f1 (Coref): 0.095
layer: 0, position: 3


acc (Coref): 18.000%, f1 (Coref): 0.163
layer: 1, position: 0


...

acc (Coref): 14.000%, f1 (Coref): 0.084
layer: 5, position: 2


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression


acc (Coref): 12.000%, f1 (Coref): 0.095
layer: 5, position: 3


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

acc (Coref): 20.000%, f1 (Coref): 0.189
Compound coreference room probing results saved to ./tutorial_data/pyvene_room_probe_coref.csv


In [ ]:
import os

if not os.path.exists('./tutorial_data'):
    os.makedirs('./tutorial_data')

df_room_probe_coref["layer"] = df_room_probe_coref["layer"].astype(int)
df_room_probe_coref["pos"] = df_room_probe_coref["position"].astype(int)
df_room_probe_coref["ACC"] = df_room_probe_coref["acc"].astype(float)

custom_labels_coref_room = [str(i) for i in range(4)]
breaks_coref_room = [0, 1, 2, 3]

plot_room_coref = (
    ggplot(df_room_probe_coref, aes(x="layer", y="pos", fill="ACC"))
    + geom_tile()
    + scale_fill_cmap("Reds") + xlab("layers")
    + scale_y_reverse(
        limits = (-0.5, 3.5),
        breaks=breaks_coref_room, labels=custom_labels_coref_room)
    + theme(figure_size=(5, 3)) + ylab("Position in Sentence")
    + theme(axis_text_y  = element_text(angle = 90, hjust = 1))
    + ggtitle("Trained Linear Probe (Compound Coreference Room Classification)")
)
ggsave(
    plot_room_coref, filename=f"./tutorial_data/pyvene_room_probe_coref.pdf", dpi=200
)
print(plot_room_coref)


<ggplot: (500 x 300)>


/usr/local/lib/python3.12/dist-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 3 in image.
/usr/local/lib/python3.12/dist-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: ./tutorial_data/pyvene_room_probe_coref.pdf
